# Dagster Hello World - Introduccion a ML Pipelines

Este notebook demuestra como usar Dagster para orquestar pipelines de datos y Machine Learning.

## Contenido
1. Configuracion del entorno
2. Conceptos basicos: Assets
3. Dependencias entre assets
4. Ejemplo de ML Pipeline
5. Ejecucion y visualizacion

## 1. Configuracion del entorno

In [ ]:
# Instalar dagster si no esta instalado
# !pip install dagster dagster-webserver
# o con uv:
# !uv add dagster dagster-webserver

In [ ]:
import dagster
from dagster import asset, op, job, Definitions, MaterializeResult, MetadataValue
from dagster import AssetExecutionContext
import pandas as pd
import numpy as np

print(f"Dagster version: {dagster.__version__}")

## 2. Conceptos Basicos: Assets

En Dagster, un **Asset** es un objeto persistente en tu sistema de datos (tabla, archivo, modelo ML, etc.).

Los assets se definen con el decorador `@asset` y representan el "que" de tu pipeline.

In [ ]:
@asset
def hello_world_asset():
    """
    Un asset simple que retorna un mensaje.
    
    Los assets son la unidad basica en Dagster.
    Representan datos persistentes en tu sistema.
    """
    message = "Hello from Dagster!"
    print(f"Ejecutando asset: {message}")
    return message

In [ ]:
@asset
def numbers_asset() -> list:
    """
    Asset que genera una lista de numeros.
    
    El type hint (-> list) ayuda a Dagster a 
    entender el tipo de dato que produce.
    """
    numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    print(f"Numeros generados: {numbers}")
    return numbers

In [ ]:
# Ejecutar assets localmente (sin servidor)
from dagster import materialize

# Materializar el asset hello_world
result = materialize([hello_world_asset])
print(f"\nResultado: {result.success}")

In [ ]:
# Materializar el asset numbers
result = materialize([numbers_asset])
print(f"\nResultado: {result.success}")

## 3. Dependencias entre Assets

Los assets pueden depender de otros assets. Dagster maneja automaticamente el orden de ejecucion.

In [ ]:
@asset
def raw_data() -> pd.DataFrame:
    """
    Asset que genera datos crudos.
    Simula la carga de datos desde una fuente externa.
    """
    np.random.seed(42)
    n_samples = 100
    
    data = pd.DataFrame({
        'feature_1': np.random.randn(n_samples),
        'feature_2': np.random.randn(n_samples),
        'feature_3': np.random.randn(n_samples),
        'target': np.random.randint(0, 2, n_samples)
    })
    
    print(f"Datos crudos generados: {len(data)} filas")
    return data

In [ ]:
@asset
def cleaned_data(raw_data: pd.DataFrame) -> pd.DataFrame:
    """
    Asset que limpia los datos.
    
    Depende de 'raw_data' - Dagster ejecuta raw_data primero.
    El nombre del parametro DEBE coincidir con el nombre del asset.
    """
    # Simular limpieza: remover outliers
    cleaned = raw_data.copy()
    
    for col in ['feature_1', 'feature_2', 'feature_3']:
        mean = cleaned[col].mean()
        std = cleaned[col].std()
        cleaned = cleaned[(cleaned[col] >= mean - 3*std) & (cleaned[col] <= mean + 3*std)]
    
    print(f"Datos limpios: {len(cleaned)} filas (removidos {len(raw_data) - len(cleaned)} outliers)")
    return cleaned

In [ ]:
@asset
def feature_statistics(cleaned_data: pd.DataFrame) -> dict:
    """
    Asset que calcula estadisticas de features.
    
    Depende de 'cleaned_data'.
    """
    stats = {}
    for col in ['feature_1', 'feature_2', 'feature_3']:
        stats[col] = {
            'mean': float(cleaned_data[col].mean()),
            'std': float(cleaned_data[col].std()),
            'min': float(cleaned_data[col].min()),
            'max': float(cleaned_data[col].max())
        }
    
    print(f"Estadisticas calculadas para {len(stats)} features")
    return stats

In [ ]:
# Materializar toda la cadena de dependencias
# Dagster automaticamente ejecuta: raw_data -> cleaned_data -> feature_statistics
result = materialize([raw_data, cleaned_data, feature_statistics])
print(f"\nPipeline ejecutado exitosamente: {result.success}")

## 4. Ejemplo de ML Pipeline

Un pipeline mas completo que incluye entrenamiento y evaluacion de un modelo.

In [ ]:
from dataclasses import dataclass
from typing import Tuple

@dataclass
class ModelMetrics:
    """Metricas del modelo."""
    accuracy: float
    precision: float
    recall: float
    f1_score: float

In [ ]:
@asset(group_name="ml_pipeline")
def training_data() -> pd.DataFrame:
    """
    Genera datos de entrenamiento para clasificacion.
    
    El parametro group_name agrupa assets relacionados en la UI.
    """
    np.random.seed(42)
    n_samples = 500
    
    # Features
    X = np.random.randn(n_samples, 5)
    
    # Target basado en una regla (para que el modelo pueda aprender)
    y = ((X[:, 0] + X[:, 1]) > 0).astype(int)
    
    # Crear DataFrame
    df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(5)])
    df['target'] = y
    
    print(f"Datos de entrenamiento: {len(df)} muestras")
    print(f"Distribucion de clases: {df['target'].value_counts().to_dict()}")
    
    return df

In [ ]:
@asset(group_name="ml_pipeline")
def train_test_split_data(training_data: pd.DataFrame) -> dict:
    """
    Divide los datos en entrenamiento y prueba.
    
    Retorna un diccionario con los conjuntos de datos.
    """
    # Split 80/20
    train_size = int(len(training_data) * 0.8)
    
    train_df = training_data.iloc[:train_size]
    test_df = training_data.iloc[train_size:]
    
    feature_cols = [col for col in training_data.columns if col != 'target']
    
    result = {
        'X_train': train_df[feature_cols].values,
        'X_test': test_df[feature_cols].values,
        'y_train': train_df['target'].values,
        'y_test': test_df['target'].values,
    }
    
    print(f"Train: {len(train_df)} | Test: {len(test_df)}")
    
    return result

In [ ]:
@asset(group_name="ml_pipeline")
def trained_model(train_test_split_data: dict) -> np.ndarray:
    """
    Entrena un modelo simple (regresion logistica manual).
    
    En un caso real, usarias sklearn, lightgbm, etc.
    """
    X_train = train_test_split_data['X_train']
    y_train = train_test_split_data['y_train']
    
    # Modelo simple: pesos aprendidos por correlacion
    n_features = X_train.shape[1]
    weights = np.zeros(n_features)
    
    # "Entrenar": calcular correlacion con target
    for i in range(n_features):
        correlation = np.corrcoef(X_train[:, i], y_train)[0, 1]
        weights[i] = correlation if not np.isnan(correlation) else 0
    
    # Normalizar pesos
    weights = weights / (np.abs(weights).sum() + 1e-8)
    
    print(f"Modelo entrenado con {n_features} features")
    print(f"Pesos: {weights.round(4)}")
    
    return weights

In [ ]:
@asset(group_name="ml_pipeline")
def model_evaluation(
    train_test_split_data: dict,
    trained_model: np.ndarray
) -> ModelMetrics:
    """
    Evalua el modelo y calcula metricas.
    
    Este asset depende de dos assets: train_test_split_data y trained_model.
    """
    X_test = train_test_split_data['X_test']
    y_test = train_test_split_data['y_test']
    weights = trained_model
    
    # Predicciones
    scores = X_test @ weights
    predictions = (scores > 0).astype(int)
    
    # Metricas
    tp = np.sum((predictions == 1) & (y_test == 1))
    tn = np.sum((predictions == 0) & (y_test == 0))
    fp = np.sum((predictions == 1) & (y_test == 0))
    fn = np.sum((predictions == 0) & (y_test == 1))
    
    accuracy = (tp + tn) / len(y_test)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    metrics = ModelMetrics(
        accuracy=round(accuracy, 4),
        precision=round(precision, 4),
        recall=round(recall, 4),
        f1_score=round(f1, 4)
    )
    
    print(f"\nMetricas del modelo:")
    print(f"  Accuracy:  {metrics.accuracy}")
    print(f"  Precision: {metrics.precision}")
    print(f"  Recall:    {metrics.recall}")
    print(f"  F1-Score:  {metrics.f1_score}")
    
    return metrics

In [ ]:
# Ejecutar el pipeline completo de ML
ml_assets = [
    training_data,
    train_test_split_data,
    trained_model,
    model_evaluation
]

result = materialize(ml_assets)
print(f"\n{'='*50}")
print(f"Pipeline ML ejecutado: {result.success}")

## 5. Definitions: Empaquetando tu Pipeline

Para desplegar en produccion, empaqueta tus assets en un objeto `Definitions`.

In [ ]:
# Crear definiciones para despliegue
defs = Definitions(
    assets=[
        # Pipeline de datos basico
        raw_data,
        cleaned_data,
        feature_statistics,
        # Pipeline de ML
        training_data,
        train_test_split_data,
        trained_model,
        model_evaluation,
    ],
)

print("Definitions creadas con", len(defs.get_asset_graph().all_asset_keys), "assets")

## 6. Ejecutar con UI Local

Para ver la UI de Dagster localmente, guarda tus definiciones en un archivo `.py` y ejecuta:

```bash
# Crear archivo de definiciones
# (copiar el codigo de assets a un archivo .py)

# Ejecutar servidor de desarrollo
dagster dev -f mi_pipeline.py

# Abrir en navegador: http://localhost:3000
```

## Resumen

En este notebook aprendimos:

1. **Assets**: Unidades basicas que representan datos persistentes
2. **Dependencias**: Dagster maneja automaticamente el orden de ejecucion
3. **Groups**: Organizar assets relacionados visualmente
4. **Materialize**: Ejecutar assets programaticamente
5. **Definitions**: Empaquetar assets para despliegue

### Ventajas de Dagster

- **Asset-centric**: Piensa en "que" produces, no solo en "como"
- **Observable**: UI rica para monitorear y debuggear
- **Testeable**: Facil de probar assets individualmente
- **Tipo-seguro**: Type hints para mejor documentacion

### Proximos pasos

- Explorar **Resources** para conexiones a bases de datos
- Usar **Schedules** para ejecucion periodica
- Implementar **Sensors** para triggers basados en eventos
- Integrar con **MLflow** para tracking de experimentos